# Ground Truth CSV → JSONL Converter

Converts a **flat CSV** ground truth file (all columns present, `NOT_FOUND` for inapplicable fields)
into a **per-type JSONL** file where each record carries only its document type's fields.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
# Set these paths before running.

CSV_INPUT  = "evaluation_data-Copy1/synthetic_original/ground_truth_synthetic.csv"
JSONL_OUTPUT = "evaluation_data-Copy1/synthetic/ground_truth.jsonl"

In [ ]:
import json
from pathlib import Path

import pandas as pd

# ── Resolve paths ──────────────────────────────────────────────────────
csv_path = Path(CSV_INPUT).expanduser().resolve()
jsonl_path = Path(JSONL_OUTPUT).expanduser().resolve()

assert csv_path.exists(), f"CSV not found: {csv_path}"

print(f"CSV input:   {csv_path}")
print(f"JSONL output: {jsonl_path}")

In [ ]:
# ── Per-type field definitions (self-contained) ──────────────────────
# Which fields belong to each document type.  Only these fields are
# carried into the JSONL output; everything else is stripped.

TYPE_FIELDS: dict[str, list[str]] = {
    "INVOICE": [
        "DOCUMENT_TYPE", "BUSINESS_ABN", "SUPPLIER_NAME", "BUSINESS_ADDRESS",
        "PAYER_NAME", "PAYER_ADDRESS", "INVOICE_DATE",
        "LINE_ITEM_DESCRIPTIONS", "LINE_ITEM_QUANTITIES",
        "LINE_ITEM_PRICES", "LINE_ITEM_TOTAL_PRICES",
        "IS_GST_INCLUDED", "GST_AMOUNT", "TOTAL_AMOUNT",
    ],
    "RECEIPT": [
        "DOCUMENT_TYPE", "BUSINESS_ABN", "SUPPLIER_NAME", "BUSINESS_ADDRESS",
        "PAYER_NAME", "PAYER_ADDRESS", "INVOICE_DATE",
        "LINE_ITEM_DESCRIPTIONS", "LINE_ITEM_QUANTITIES",
        "LINE_ITEM_PRICES", "LINE_ITEM_TOTAL_PRICES",
        "IS_GST_INCLUDED", "GST_AMOUNT", "TOTAL_AMOUNT",
    ],
    "BANK_STATEMENT": [
        "DOCUMENT_TYPE", "STATEMENT_DATE_RANGE",
        "LINE_ITEM_DESCRIPTIONS", "TRANSACTION_DATES",
        "TRANSACTION_AMOUNTS_PAID",
    ],
    "TRAVEL": [
        "DOCUMENT_TYPE", "PASSENGER_NAME", "TRAVEL_MODE", "TRAVEL_ROUTE",
        "TRAVEL_DATES", "INVOICE_DATE", "GST_AMOUNT", "TOTAL_AMOUNT",
        "SUPPLIER_NAME",
    ],
    "TRANSACTION_LINK": [
        "DOCUMENT_TYPE", "BANK_STATEMENT_FILE",
        "RECEIPT_DATE", "RECEIPT_DESCRIPTION", "RECEIPT_TOTAL",
        "BANK_TRANSACTION_DATE", "BANK_TRANSACTION_DESCRIPTION",
        "BANK_TRANSACTION_DEBIT",
    ],
    "LOGBOOK": [
        "DOCUMENT_TYPE", "VEHICLE_MAKE", "VEHICLE_MODEL",
        "VEHICLE_REGISTRATION", "ENGINE_CAPACITY",
        "LOGBOOK_PERIOD_START", "LOGBOOK_PERIOD_END",
        "ODOMETER_START", "ODOMETER_END",
        "TOTAL_KILOMETERS", "BUSINESS_KILOMETERS",
        "BUSINESS_USE_PERCENTAGE",
        "JOURNEY_DATES", "JOURNEY_DISTANCES", "JOURNEY_PURPOSES",
    ],
}

print("Document types and field counts:")
for dt, fields in sorted(TYPE_FIELDS.items()):
    print(f"  {dt:20s} → {len(fields)} fields")

In [ ]:
# ── Read CSV ───────────────────────────────────────────────────────────
df = pd.read_csv(csv_path, dtype=str).fillna("NOT_FOUND")

# First column is always the filename; DOCUMENT_TYPE is the type column.
filename_col = df.columns[0]
assert "DOCUMENT_TYPE" in df.columns, (
    f"No DOCUMENT_TYPE column found. Columns present: {list(df.columns)}"
)
doc_type_col = "DOCUMENT_TYPE"

print(f"Rows: {len(df)}  |  Filename column: '{filename_col}'  |  Type column: '{doc_type_col}'")
print("\nDocument type distribution:")
print(df[doc_type_col].value_counts().to_string())
print("\nCSV preview (first 3 rows):")
df.head(3)

In [ ]:
# ── Convert to JSONL ──────────────────────────────────────────────────
records_written = 0
skipped_types: dict[str, int] = {}
all_records: list[dict[str, str]] = []

for _, row in df.iterrows():
    doc_type = str(row[doc_type_col]).strip().upper()
    filename = str(row[filename_col]).strip()

    # Look up per-type fields
    fields = TYPE_FIELDS.get(doc_type)
    if fields is None:
        skipped_types[doc_type] = skipped_types.get(doc_type, 0) + 1
        continue

    record: dict[str, str] = {"filename": filename}
    for field in fields:
        val = str(row.get(field, "NOT_FOUND")).strip()
        record[field] = val

    all_records.append(record)
    records_written += 1

# Write output
jsonl_path.parent.mkdir(parents=True, exist_ok=True)
with jsonl_path.open("w") as f:
    for rec in all_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Wrote {records_written} records to {jsonl_path}")
if skipped_types:
    for dtype, count in sorted(skipped_types.items()):
        print(f"  SKIPPED {count} records with unknown type: {dtype}")

In [ ]:
# ── Preview output ────────────────────────────────────────────────────
print("Output JSONL preview (first 3 records):\n")
for rec in all_records[:3]:
    print(json.dumps(rec, indent=2, ensure_ascii=False))
    print()